# S5 - steering + consolidated manifest + S6 judge

Target this session at **240-270 min** wall clock; hard boundary **300 min**. Do not start a stage/condition that the calibrated projection says cannot finish (analysis_plan.md §7).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'main'
PINNED_COMMIT = '37c22f4fa0d8a1098017f5518cdeb0b7ad4cf5dd'   # exact commit this session runs against

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit
# loads (see 04b). Matches colab_unified_{analysis,training}.ipynb.
!pip uninstall -y torchao || true
!nvidia-smi

## 4. Persistent storage (results/ + HF cache bound to Drive)

In [ ]:
# Bind results/ + the HF weight cache to a persistent Drive folder so this
# session's work survives a Colab disconnect and a fresh VM resumes it. One
# line - the logic + tests live in src/colab_persist.py. Idempotent, and safe
# even if you have already done work on the ephemeral results/ (it merges that
# into Drive first). Override the Drive root with the DPO_DRIVE_ROOT env var;
# pass persist_hf_cache=False to keep the ~5 GB HF cache off Drive.
from src.colab_persist import bind, status_line
info = bind()                     # or: bind(persist_hf_cache=False)
print(status_line(info))
!python -m src.analysis.v2_pipeline status

## 5. Steering: learned vs random, dose-response {0.5, 1.0, 2.0}

In [ ]:
!python -m src.analysis.v2_pipeline steering --stage M3 --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_alt --alpha-coefficients 0.5 1.0 2.0

## 6. If tight: cut M1/M2 dose-response FIRST (never the random control)

In [ ]:
from src.analysis.intervention_conditions import steering_cut_order
print(steering_cut_order())

## 7. Build the consolidated response manifest (AFTER S2 + S4 + S5)

In [ ]:
import datetime
ts = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
# fill in the per-session manifest paths + the frozen SHAs from S0's load_run_inputs()
!python -m src.analysis.behavioral_judges --response-manifest results/manifests/consolidated_{ts}.json --build-consolidated results/manifests/<s2>.json results/manifests/<s4>.json results/manifests/<s5>.json --benchmark-sha256 <BENCH_SHA> --split-manifest-sha256 <SPLIT_SHA>

## 8. S6 judge pass - consumes ONLY the consolidated manifest

In [ ]:
!python -m src.analysis.behavioral_judges --response-manifest results/manifests/consolidated_{ts}.json --require-binding --reject-legacy --out-dir results/behavioral_judges_v2 --run-live

## 9. Post-run: bridge outputs, re-validate, session summary

In [ ]:
!python -m src.analysis.verify_activations
!python -m src.analysis.v2_pipeline status